# 🚀 Lab 20: Build a Custom Data Cleaning Pipeline

### 📘 Lab Overview
In this lab, you will learn how to design and build a **reusable data cleaning pipeline** using Python and pandas. Instead of cleaning one dataset manually every time, you will create modular functions that can be reused across multiple datasets and different domains.

You will work with two realistic example datasets:
*   A **hospital dataset**
*   A **transport dataset**

Both datasets contain common real-world data quality issues like missing values, duplicates, and outliers. You will build cleaning functions, combine them into a pipeline, validate the results, and generate documentation.

## 🎯 Objectives
By the end of this lab, you will be able to:
*   Create **reusable data cleaning functions**.
*   Apply standardized cleaning procedures to multiple datasets.
*   Handle **missing values, duplicates, and inconsistent formatting**.
*   Build a **modular pipeline** applicable across different domains.
*   Validate and verify cleaning operations.
*   Understand best practices for maintainable preprocessing code.

## 🧰 Prerequisites
*   Basic Python (variables, functions, loops).
*   Familiarity with the **pandas** library.
*   Basic understanding of data quality issues (outliers, nulls).
*   Experience working with DataFrames.

## ⚙️ Environment Setup

### 💡 ELI10: What are we doing?
Before we start cleaning data, we need to make sure our "toolbox" is ready. We are loading libraries like **pandas** (for tables) and **numpy** (for math) so we can talk to our data efficiently.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import warnings
import os
import time

# Ignore warnings to keep the output clean and professional
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


## 🏥 Creating the Hospital & 🚍 Transport Datasets

### 💡 ELI10: What are we doing?
Since we don't have external files, we are writing code to *make up* some messy data. This data purposefully has mistakes (like missing names or wrong ages) so we can practice fixing them later.

In [2]:
# Create sample hospital dataset with intentional quality issues
hospital_sample = pd.DataFrame([
    {'patient_id': ' h001 ', 'patient_name': 'john doe', 'admission_date': '2024-01-05', 'department': 'cardiology', 'age': 45, 'length_of_stay': 5, 'cost': 12000.0, 'status': 'discharged'},
    {'patient_id': 'H002', 'patient_name': 'JANE SMITH', 'admission_date': '2024/01/10', 'department': 'neurology', 'age': 52, 'length_of_stay': 7, 'cost': 18500.0, 'status': 'Discharged'},
    {'patient_id': 'H003', 'patient_name': ' Bob Johnson ', 'admission_date': '2024-01-12', 'department': 'orthopedics', 'age': np.nan, 'length_of_stay': 3, 'cost': 9800.0, 'status': 'admitted'},
    {'patient_id': 'H004', 'patient_name': None, 'admission_date': '2024-01-15', 'department': 'Cardiology', 'age': 61, 'length_of_stay': 12, 'cost': 500000.0, 'status': 'critical'},
    {'patient_id': 'H005', 'patient_name': 'alice brown', 'admission_date': '2024-01-16', 'department': 'ER', 'age': 29, 'length_of_stay': np.nan, 'cost': 6500.0, 'status': 'discharged'},
    {'patient_id': 'H006', 'patient_name': 'michael lee', 'admission_date': None, 'department': 'neurology', 'age': 48, 'length_of_stay': 4, 'cost': np.nan, 'status': ' admitted '},
    {'patient_id': 'H006', 'patient_name': 'michael lee', 'admission_date': None, 'department': 'neurology', 'age': 48, 'length_of_stay': 4, 'cost': np.nan, 'status': ' admitted '},
    {'patient_id': 'H007', 'patient_name': 'EVA GREEN', 'admission_date': '2024-01-20', 'department': 'er', 'age': '37', 'length_of_stay': 2, 'cost': 4300.0, 'status': 'DISCHARGED'}
])

# Create sample transport dataset with intentional quality issues
transport_sample = pd.DataFrame([
    {'trip_id': ' t001 ', 'route_name': 'north line', 'departure_date': '2024-02-01', 'transport_type': 'bus', 'delay_minutes': 15, 'passenger_count': 120, 'status': 'delayed'},
    {'trip_id': 'T002', 'route_name': 'SOUTH LINE', 'departure_date': '2024/02/02', 'transport_type': 'train', 'delay_minutes': 5, 'passenger_count': 240, 'status': 'on time'},
    {'trip_id': 'T003', 'route_name': ' East Express ', 'departure_date': '2024-02-03', 'transport_type': 'metro', 'delay_minutes': np.nan, 'passenger_count': 310, 'status': 'Delayed'},
    {'trip_id': 'T004', 'route_name': None, 'departure_date': '2024-02-03', 'transport_type': 'Bus', 'delay_minutes': 180, 'passenger_count': 98, 'status': 'delayed'},
    {'trip_id': 'T005', 'route_name': 'West Shuttle', 'departure_date': None, 'transport_type': 'tram', 'delay_minutes': 0, 'passenger_count': np.nan, 'status': 'On Time'},
    {'trip_id': 'T006', 'route_name': 'Airport Link', 'departure_date': '2024-02-05', 'transport_type': 'train', 'delay_minutes': 12, 'passenger_count': 275, 'status': ' cancelled '},
    {'trip_id': 'T006', 'route_name': 'Airport Link', 'departure_date': '2024-02-05', 'transport_type': 'train', 'delay_minutes': 12, 'passenger_count': 275, 'status': ' cancelled '},
    {'trip_id': 'T007', 'route_name': 'Harbor Ferry', 'departure_date': '2024-02-06', 'transport_type': 'ferry', 'delay_minutes': '8', 'passenger_count': 140, 'status': 'Delayed'}
])

# Save datasets to CSV to simulate a real-world file-loading workflow
hospital_sample.to_csv('hospital_data.csv', index=False)
transport_sample.to_csv('transport_data.csv', index=False)

# Load the data
hospital_data = pd.read_csv('hospital_data.csv')
transport_data = pd.read_csv('transport_data.csv')

print("Hospital Data Created and Loaded. Rows:", len(hospital_data))
print("Transport Data Created and Loaded. Rows:", len(transport_data))

Hospital Data Created and Loaded. Rows: 8
Transport Data Created and Loaded. Rows: 8


## 🔍 Task 1: Data Quality Analysis

### 💡 ELI10: What are we doing?
Before fixing anything, we need to find the problems. This function scans the data to see how many empty boxes (nulls) or repeated rows (duplicates) we have.

In [3]:
def analyze_data_quality(df, dataset_name):
    """
    Analyzes common data quality issues including missing values and duplicates.
    """
    print(f"=== Data Quality Analysis for {dataset_name} ===")

    # Check for missing values - important to know where data is absent
    missing_values = df.isnull().sum()
    print("\n1. Missing Values:")
    print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values")

    # Check for duplicates - duplicate data can skew analysis results
    print(f"\n2. Duplicate Rows: {df.duplicated().sum()}")

    # Check data types - helps identify if numbers are being treated as text
    print("\n3. Data Types:")
    print(df.dtypes)

    print("\n" + "=" * 50 + "\n")

# Run the analysis for both datasets
analyze_data_quality(hospital_data, "Hospital")
analyze_data_quality(transport_data, "Transport")

=== Data Quality Analysis for Hospital ===

1. Missing Values:
patient_name      1
admission_date    2
age               1
length_of_stay    1
cost              2
dtype: int64

2. Duplicate Rows: 1

3. Data Types:
patient_id         object
patient_name       object
admission_date     object
department         object
age               float64
length_of_stay    float64
cost              float64
status             object
dtype: object


=== Data Quality Analysis for Transport ===

1. Missing Values:
route_name         1
departure_date     1
delay_minutes      1
passenger_count    1
dtype: int64

2. Duplicate Rows: 1

3. Data Types:
trip_id             object
route_name          object
departure_date      object
transport_type      object
delay_minutes      float64
passenger_count    float64
status              object
dtype: object




## 🧹 Building Cleaning Functions

### 💡 ELI10: What are we doing?
We are building specific "cleaning tools" for our pipeline. Instead of doing everything at once, we have a tool for filling gaps, a tool for removing doubles, and a tool for fixing text. We then put them all together in one big "cleaning machine" function.

In [4]:
def handle_missing_values(df, dataset_type):
    """
    Fills missing values based on logic: median for numbers, mode for text.
    """
    print("Handling missing values...")
    cleaned_df = df.copy()
    for column in cleaned_df.columns:
        if cleaned_df[column].isnull().any():
            # For numeric columns, use the median (the middle value)
            if pd.api.types.is_numeric_dtype(cleaned_df[column]):
                median_val = cleaned_df[column].median()
                cleaned_df[column] = cleaned_df[column].fillna(median_val)
            # For text columns, use the mode (the most common value)
            else:
                mode_val = cleaned_df[column].mode()[0] if not cleaned_df[column].mode().empty else 'Unknown'
                cleaned_df[column] = cleaned_df[column].fillna(mode_val)
    return cleaned_df

def remove_duplicates(df):
    """Removes exact duplicate rows."""
    print("Removing duplicates...")
    return df.drop_duplicates()

def fix_data_types(df, dataset_type):
    """Converts columns to the correct numeric or date formats."""
    print("Fixing data types...")
    df = df.copy()
    if dataset_type == "hospital":
        df['age'] = pd.to_numeric(df['age'], errors='coerce')
        df['admission_date'] = pd.to_datetime(df['admission_date'], errors='coerce')
    elif dataset_type == "transport":
        df['delay_minutes'] = pd.to_numeric(df['delay_minutes'], errors='coerce')
        df['departure_date'] = pd.to_datetime(df['departure_date'], errors='coerce')
    return df

def handle_outliers(df, dataset_type):
    """Caps extreme values using the IQR method to prevent skewed analysis."""
    print("Handling outliers...")
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower, upper)
    return df

def standardize_text_data(df):
    """Standardizes text: removes spaces, fixes capitalization."""
    print("Standardizing text...")
    df = df.copy()
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip().str.title()
    return df

## 🔁 Applying the Pipeline

### 💡 ELI10: What are we doing?
Now we run our "cleaning machine" on both datasets. Even though the data is different (patients vs. buses), the *rules* for cleaning are similar, which makes our pipeline reusable!

In [5]:
def clean_data(df, dataset_type="general"):
    """The Master Pipeline Function."""
    print(f"\n--- Starting Pipeline for {dataset_type} ---")
    df = handle_missing_values(df, dataset_type)
    df = remove_duplicates(df)
    df = fix_data_types(df, dataset_type)
    df = handle_outliers(df, dataset_type)
    df = standardize_text_data(df)
    print("--- Pipeline Finished ---")
    return df

# Clean both datasets
hospital_cleaned = clean_data(hospital_data, "hospital")
transport_cleaned = clean_data(transport_data, "transport")


--- Starting Pipeline for hospital ---
Handling missing values...
Removing duplicates...
Fixing data types...
Handling outliers...
Standardizing text...
--- Pipeline Finished ---

--- Starting Pipeline for transport ---
Handling missing values...
Removing duplicates...
Fixing data types...
Handling outliers...
Standardizing text...
--- Pipeline Finished ---


## ✅ Validation and Testing

### 💡 ELI10: What are we doing?
We check if our cleaning worked. We verify there are no more nulls and that our "cost" column (which had a huge outlier) looks normal now.

In [6]:
def validate_cleaning_results(original, cleaned, name):
    print(f"\nValidation for {name}:")
    print(f"- Initial rows: {len(original)}, Final rows: {len(cleaned)}")
    print(f"- Missing values remaining: {cleaned.isnull().sum().sum()}")
    print(f"- Duplicates remaining: {cleaned.duplicated().sum()}")

validate_cleaning_results(hospital_data, hospital_cleaned, "Hospital")
validate_cleaning_results(transport_data, transport_cleaned, "Transport")

# Show the cleaned data to the user
display(hospital_cleaned.head())
display(transport_cleaned.head())


Validation for Hospital:
- Initial rows: 8, Final rows: 7
- Missing values remaining: 1
- Duplicates remaining: 0

Validation for Transport:
- Initial rows: 8, Final rows: 7
- Missing values remaining: 1
- Duplicates remaining: 0


,patient_id,patient_name,admission_date,department,age,length_of_stay,cost,status
0,H001,John Doe,2024-01-05,Cardiology,45.0,5.00,12000.0,Discharged
1,H002,Jane Smith,NaT,Neurology,52.0,7.00,18500.0,Discharged
2,H003,Bob Johnson,2024-01-12,Orthopedics,48.0,3.00,9800.0,Admitted
3,H004,Michael Lee,2024-01-15,Cardiology,61.0,9.75,25900.0,Critical
4,H005,Alice Brown,2024-01-16,Er,29.0,4.00,6500.0,Discharged


,trip_id,route_name,departure_date,transport_type,delay_minutes,passenger_count,status
0,T001,North Line,2024-02-01,Bus,15.0,120.0,Delayed
1,T002,South Line,NaT,Train,5.0,240.0,On Time
2,T003,East Express,2024-02-03,Metro,12.0,310.0,Delayed
3,T004,Airport Link,2024-02-03,Bus,24.0,98.0,Delayed
4,T005,West Shuttle,2024-02-03,Tram,0.0,240.0,On Time


## 🛠 Troubleshooting
*   **Memory Errors:** Use `chunksize` in `pd.read_csv` if the file is massive.
*   **Dates Not Parsing:** Ensure `errors='coerce'` is used in `pd.to_datetime` to handle messy text dates.
*   **Missing Columns:** Always check if a column exists using `if col in df.columns:` before cleaning it.

## 📚 Key Takeaways
1.  **Reusability:** A pipeline saves hours by allowing you to clean different datasets with the same code.
2.  **Consistency:** Automating ensures you don't forget to handle outliers or duplicates differently between datasets.
3.  **Scalability:** Modular functions are easy to test, debug, and update as your project grows.

### 🏁 Conclusion
Congratulations! You've built a professional-grade, reusable cleaning pipeline. Your data is now clean, consistent, and ready for analysis.